In [1]:
import random
from pathlib import Path
from collections import defaultdict
from typing import Optional
 
import torch
from torch.utils.data import Dataset, DataLoader
LIBRE_TTS_PATH = '../data/processed/libreTTS/'
UA_PATH = '../data/processed/ua/'

# raw wavs, for ECAPA speaker embeddings (the *_PATH dirs above hold .pt mels only)
LIBRE_TTS_RAW_PATH = '../data/LibriTTS/train-clean-100/'
UA_RAW_PATH = '../data/Noise Reduced UASPEECH Dysarthria Dataset/noisereduced-uaspeech/'
UA_CONTROL_RAW_PATH = '../data/Noise Reduced UASPEECH Dysarthria Dataset/noisereduced-uaspeech-control/'


In [2]:
class LibreTTSDataset(Dataset):
    def __init__(self):
        super().__init__()
        self.samples = []
        self.speakers = set()
        self.max_frames = 860
        self.min_frames = 86
        self.raw_audio_path = LIBRE_TTS_RAW_PATH
         
        
        print(Path(LIBRE_TTS_PATH).rglob("*.pt"))

        for pt_file in sorted(Path(LIBRE_TTS_PATH).rglob("*.pt")):

            self.samples.append({
                "path":str(pt_file),
                "speaker_id": pt_file.parent.name
            })

            self.speakers.add(pt_file.parent.name)

    
    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
 
        data = torch.load(sample["path"], weights_only=False)
        mel = data["mel"]  # (80, T)
 
        n_frames = mel.shape[-1]
 
        if n_frames < self.min_frames:
            return self.__getitem__((idx + 1) % len(self))
 
        if n_frames > self.max_frames:
            start = random.randint(0, n_frames - self.max_frames)
            mel = mel[:, start:start + self.max_frames]
            n_frames = self.max_frames
 
        return {
            "input_mel": mel,           # (80, T)
            "target_mel": mel.clone(),  # same — self-reconstruction
            "speaker_id": sample["speaker_id"],
            "input_length": n_frames,
            "target_length": n_frames,
        }



In [3]:
def collate_fn(batch):
    input_mels = [b["input_mel"] for b in batch]
    target_mels = [b["target_mel"] for b in batch]
    target_lengths = [b["target_length"] for b in batch]
    input_lengths = [b["input_length"] for b in batch]
    
    max_input_len = max(input_lengths)
    max_target_len = max(target_lengths)
    
    
    padded_inputs = [] 
    input_masks = []
    for mel,len in zip(input_mels,input_lengths):
        n_mels,T = mel.shape
        
        if T < max_input_len:
            mel = torch.concat([mel,torch.zeros(n_mels,max_input_len-T)],dim=-1)

        padded_inputs.append(mel)
        mask = torch.zeros(max_input_len,dtype=torch.bool)
        mask[len:] = True
        input_masks.append(mask)    

    padded_targets = [] 
    target_masks = []
    for mel,len in zip(target_mels,target_lengths):
        n_mels,T = mel.shape
        
        if T < max_target_len:
            mel = torch.concat([mel,torch.zeros(n_mels,max_target_len-T)],dim=-1)

        padded_targets.append(mel)
        mask = torch.zeros(max_target_len,dtype=torch.bool)
        mask[len:] = True
        target_masks.append(mask)
        
    out = {
         "input_mel": torch.stack(padded_inputs),         
        "target_mel": torch.stack(padded_targets),       
        "input_lengths": torch.tensor(input_lengths),    
        "target_lengths": torch.tensor(target_lengths),  
        "padding_mask": torch.stack(input_masks),        
        "target_mask": torch.stack(target_masks),        
        "speaker_ids": [b["speaker_id"] for b in batch],
    }

    # Recognizer targets: CF02's code indices (T', groups) at the encoder's half rate.
    # Padded with -100 so cross_entropy ignores them. (len() is shadowed in this function.)
    # Multi-reference targets (UASpeechDataset(multi_ref=True)) are (T', groups, refs) and become
    # (B, T', groups, R); a file without them contributes its single CF02 code as ref 0.
    if "target_codes" in batch[0]:
        n_items = batch.__len__()
        Tp = (max_input_len + 1) // 2
        G = batch[0]["target_codes"].shape[1]
        if all(b["target_codes"].dim() == 2 for b in batch):
            codes = torch.full((n_items, Tp, G), -100, dtype=torch.long)
            for i, b in enumerate(batch):
                c = b["target_codes"][:Tp]
                codes[i, :c.shape[0]] = c
        else:
            R = max(b["target_codes"].shape[2] if b["target_codes"].dim() == 3 else 1 for b in batch)
            codes = torch.full((n_items, Tp, G, R), -100, dtype=torch.long)
            for i, b in enumerate(batch):
                c = b["target_codes"][:Tp]
                if c.dim() == 2:
                    c = c.unsqueeze(-1)
                codes[i, :c.shape[0], :, :c.shape[-1]] = c
        out["target_codes"] = codes

    # Tokenizer consistency pairs (Codes2MelDataset(with_pairs=True)): a second recording of the same
    # word and, per input frame, the index of its DTW-aligned frame in that recording (-1 = no pair).
    if any("pair_mel" in b for b in batch):
        n_items = batch.__len__()
        n_bins = batch[0]["input_mel"].shape[0]
        Tq = max(b["pair_mel"].shape[-1] for b in batch if "pair_mel" in b)
        pair = torch.zeros(n_items, n_bins, Tq)
        pair_mask = torch.ones(n_items, Tq, dtype=torch.bool)
        pair_index = torch.full((n_items, max_input_len), -1, dtype=torch.long)
        for i, b in enumerate(batch):
            if "pair_mel" in b:
                m = b["pair_mel"]
                pair[i, :, :m.shape[-1]] = m
                pair_mask[i, :m.shape[-1]] = False
                ix = b["pair_index"][:max_input_len]
                pair_index[i, :ix.shape[0]] = ix
        out.update(pair_mel=pair, pair_padding_mask=pair_mask, pair_index=pair_index)
    return out
        

if __name__ == "__main__":
    dl = DataLoader(
        LibreTTSDataset(),
        batch_size=2,
        shuffle=True,
        num_workers=4,
        drop_last=True,
        collate_fn=collate_fn
    )
    

    data = next(iter(dl))
    assert data['input_mel'].shape[0] == 2
    assert data['padding_mask'].shape[1] == data['input_mel'].shape[-1]
    print("LibreTTS ok:", data['input_mel'].shape)

<generator object Path.rglob at 0x786ad0f2e020>
LibreTTS ok: torch.Size([2, 80, 860])


## UASpeech

In [ ]:
SEVERITY = {
    "M01": "low",     "M04": "low",     "M05": "mid",
    "M07": "mid",     "M08": "mid",     "M09": "mid",
    "M10": "high",    "M11": "low",     "M12": "high",
    "M14": "mid",
    "F02": "low",     "F03": "mid",     "F04": "mid",
    "F05": "high",
}
 
DYSARTHRIC_SPEAKERS = set(SEVERITY.keys())
 
CONTROL_SPEAKERS = {
    "CM01", "CM04", "CM05", "CM06", "CM08", "CM09",
    "CM10", "CM12", "CM13",
    "CF02", "CF03", "CF04", "CF05",
}

REFERENCE_CONTROL = "CF02" # targets are DTW-aligned to this speaker in preprocess.ipynb

FINETUNE_BLOCKS = ['B1','B3'] # There are 3 blocks in the dataset, 2 for training, 1 for eval

In [ ]:
class UASpeechDataset(Dataset):
    def __init__(self, multi_ref=False):
        super().__init__()
        self.max_frames = 860
        # True: recognizer targets from several reference speakers' takes ("target_codes_multi",
        # precompute_multi_target_codes in main.ipynb) instead of CF02's take alone
        self.multi_ref = multi_ref
        
        self.word_to_dysarthric = defaultdict(list)
        self.word_to_control = defaultdict(list)
        self.speakers = set()
        
        self.raw_audio_path = UA_RAW_PATH
        # speaker/word live in the filename (SPK_B1_CW75_M8.pt), so the index
        # builds without torch.load-ing all ~136k shards.
        for pt_file in sorted(Path(UA_PATH).rglob("*.pt")):
            
            parts = pt_file.stem.split("_")
            if len(parts) != 4:
                continue
            speaker_id, block, word_id, _mic = parts
            word = f"{block}_{word_id}"
            
            if(block not in FINETUNE_BLOCKS):
                continue
            
            if(speaker_id in DYSARTHRIC_SPEAKERS):
                self.speakers.add(speaker_id)
                self.word_to_dysarthric[word].append((speaker_id,str(pt_file)))
                
            elif(speaker_id in CONTROL_SPEAKERS):
                self.word_to_control[word].append((speaker_id,str(pt_file)))
                
                
    
        self.pairs = [];
        
        for word in self.word_to_dysarthric:
            
            control_options = self.word_to_control.get(word, [])
            if not control_options:      # no control recording of this word -> no target
                continue
            
            options = dict(control_options)
            if REFERENCE_CONTROL not in options:
                continue
            
            for dys_speaker, dys_path in self.word_to_dysarthric[word]:
                self.pairs.append({
                    "dysarthric_path": dys_path,
                    "dysarthric_speaker": dys_speaker,
                    "word":word,
                    "severity":SEVERITY.get(dys_speaker),
                    "control_options": control_options,
                })

        if self.multi_ref and self.pairs:
            probe = torch.load(self.pairs[0]["dysarthric_path"], weights_only=False)
            if "target_codes_multi" not in probe:
                raise RuntimeError("multi_ref=True but no target_codes_multi in the UA files: "
                                   "run precompute_multi_target_codes(...) in main.ipynb first")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        pair = self.pairs[index]
        data = torch.load(pair["dysarthric_path"], weights_only=False)
        input_mel = data["mel"]
        # KeyError here = run the DTW cell at the end of preprocess.ipynb
        target_mel = data["dtw_target"]   # REFERENCE_CONTROL's take, aligned frame-by-frame to input_mel
        
 
        
     
      
        
        
        
        if input_mel.shape[-1] > self.max_frames:
            input_mel = input_mel[:, :self.max_frames]
        if target_mel.shape[-1] > self.max_frames:
            target_mel = target_mel[:, :self.max_frames]

        # Recognizer targets (precompute_target_codes in main.ipynb): one code per group per
        # encoder frame, i.e. ceil(T/2) frames for T input frames
        target_codes = None
        if "target_codes" in data:
            tc = data["target_codes_multi"] if self.multi_ref and "target_codes_multi" in data else data["target_codes"]
            target_codes = torch.as_tensor(tc).long()[: (input_mel.shape[-1] + 1) // 2]   # (T',G) or (T',G,R)

        
            
            
            
            
        
        
        return {
            "input_mel": input_mel,
            "target_mel": target_mel,
            "speaker_id": pair["dysarthric_speaker"],
            "input_length": input_mel.shape[-1],
            "target_length": target_mel.shape[-1],
            "word": pair["word"],
            "severity": pair["severity"],
            "control_speaker": REFERENCE_CONTROL,
            **({"target_codes": target_codes} if target_codes is not None else {}),
        }
            
          

if __name__ == "__main__":
    dl = DataLoader(
        UASpeechDataset(),
        batch_size=2,
        shuffle=True,
        num_workers=4,
        drop_last=True,
        collate_fn=collate_fn
    )

    b = next(iter(dl))
    # input and target are different utterances, so they pad to different lengths
    assert b['target_mask'].shape[1] == b['target_mel'].shape[-1]
    assert b['padding_mask'].shape[1] == b['input_mel'].shape[-1]
    assert set(b['speaker_ids']) <= DYSARTHRIC_SPEAKERS
    print("UASpeech ok:", len(dl.dataset), "pairs,", len(dl.dataset.speakers), "speakers")


TypeError: float() argument must be a string or a real number, not 'dict'